# Exploración inicial de datos ENAHO

Reconocer los archivos y diccionarios disponibles, sus formatos y estructura general.

## Pasos previstos
- Inventariar los archivos locales disponibles sin publicar datos.
- Revisar documentación y diccionarios asociados.
- Registrar preguntas y decisiones pendientes sobre cobertura, unidades y módulos.

No se incluyen datos ni resultados. La variable objetivo y los nombres de variables ENAHO se decidirán tras revisar los archivos y diccionarios originales.

In [45]:
%pip install pandas
%pip install openpyxl

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip



   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   ---------------------------------------- 2/2 [openpyxl]

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [18]:
import re
import sys
import unicodedata
from pathlib import Path
import pandas as pd

Ubico los archivos en el directorio 

In [35]:
RAIZ = Path.cwd().resolve().parent
RAW = RAIZ / "data" / "raw"
INTERIM = RAIZ / "data" / "interim"
INTERIM.mkdir(parents=True, exist_ok=True)
 

ARCHIVOS = {
    "sumaria": "Sumaria-2025.csv",
    "m100": "Enaho01-2025-100.csv",
    "m200": "Enaho01-2025-200.csv",
    "m300": "Enaho01A-2025-300.csv",
}
# Módulos que están a nivel persona (llevan CODPERSO)
A_NIVEL_PERSONA = {"m200", "m300"}
 
# Largo oficial de cada llave según el diccionario ENAHO (para recuperar ceros a la izquierda)
LARGO_LLAVE = {"conglome": 6, "vivienda": 3, "hogar": 2, "codperso": 2}
print(RAIZ)

C:\Users\LENOVO\PROYECTOS\modelo-prediccion-pobreza-enaho


Funcionalidades 

In [36]:
def normalizar_nombre(col: str) -> str:
    """'AÑO' -> 'ano', 'P25$1' -> 'p25_1', 'NCONGLOME' -> 'nconglome'."""
    col = unicodedata.normalize("NFKD", col).encode("ascii", "ignore").decode()
    return col.strip().lower().replace("$", "_")
 
 
def es_columna_monetaria_derivada(col_normalizada: str) -> bool:
    """Columnas D+número (deflactadas) e I+número (imputadas): son montos de gasto/ingreso
    y entran al cálculo de la pobreza. Se descartan desde la lectura (fuga de información)."""
    return re.match(r"^[di]\d", col_normalizada) is not None


def detectar_encoding(ruta: Path) -> str:
    """Los CSV del INEI suelen venir en Latin-1 (la Ñ de AÑO sale como '�' si se lee como UTF-8)."""
    with open(ruta, "rb") as f:
        primera_linea = f.readline()
    try:
        primera_linea.decode("utf-8")
        return "utf-8-sig"
    except UnicodeDecodeError:
        return "latin-1"


def leer_csv(ruta: Path, descartar_montos: bool = True) -> pd.DataFrame:
    if not ruta.exists():
        disponibles = "\n  ".join(sorted(p.name for p in ruta.parent.iterdir()))
        sys.exit(f"No encuentro {ruta.name} en {ruta.parent}.\nArchivos que sí hay:\n  {disponibles}\n"
                 f"Ajusta el diccionario ARCHIVOS al inicio del script.")
    def usar(col):
            c = normalizar_nombre(col)
            if c.startswith("unnamed") or c == "":
                return False  # columna vacía por el ';' final de cada línea
            return not (descartar_montos and es_columna_monetaria_derivada(c))
    df = pd.read_csv(
            ruta,
            sep=";",
            dtype=str,              # TODO como texto: así no se pierden los ceros de las llaves
            encoding=detectar_encoding(ruta),
            usecols=usar,
            keep_default_na=False,  # los vacíos llegan como "" y los limpiamos abajo
            low_memory=False,
    )
    df.columns = [normalizar_nombre(c) for c in df.columns]
    if df.columns.duplicated().any():
            raise ValueError(f"{ruta.name}: hay nombres de columna repetidos tras normalizar: "
                            f"{df.columns[df.columns.duplicated()].tolist()}")
    
    # ENAHO deja los vacíos como " " (espacio): los convertimos a NaN real
    for c in df.columns:
            s = df[c].str.strip()
            df[c] = s.where(s != "", pd.NA)
    return df


def crear_llaves(df: pd.DataFrame, nombre: str) -> pd.DataFrame:
    faltan = [k for k in ("conglome", "vivienda", "hogar") if k not in df.columns]
    if faltan:
        raise KeyError(f"{nombre}: faltan las columnas {faltan}")
    for col, largo in LARGO_LLAVE.items():
        if col in df.columns:
            if df[col].isna().any():
                raise ValueError(f"{nombre}: la llave '{col}' tiene valores vacíos")
            df[col] = df[col].str.zfill(largo)
    df["id_hogar"] = df["conglome"] + df["vivienda"] + df["hogar"]
    if nombre in A_NIVEL_PERSONA:
        df["id_persona"] = df["id_hogar"] + df["codperso"]
    return df
 
 
def guardar(df: pd.DataFrame, nombre: str) -> Path:
    try:
        ruta = INTERIM / f"{nombre}.parquet"
        df.to_parquet(ruta, index=False)
    except ImportError:  # no tienes pyarrow: usa pickle (equivalente para este proyecto)
        ruta = INTERIM / f"{nombre}.pkl"
        df.to_pickle(ruta)
    return ruta

In [41]:
def main():
    tablas = {}
    for nombre, archivo in ARCHIVOS.items():
        print(f"Leyendo {archivo} ...")
        df = crear_llaves(leer_csv(RAW / archivo), nombre)
        tablas[nombre] = df
 
        unica = "id_persona" if nombre in A_NIVEL_PERSONA else "id_hogar"
        repetidas = int(df[unica].duplicated().sum())
        print(f"  {len(df):,} filas x {df.shape[1]} columnas | hogares distintos: {df['id_hogar'].nunique():,}")
        print(f"  '{unica}' repetidos: {repetidas}" + ("  <-- REVISAR" if repetidas else "  (ok, único)"))
 
    sumaria, m100 = tablas["sumaria"], tablas["m100"]
    ids_sum, ids_100 = set(sumaria["id_hogar"]), set(m100["id_hogar"])
 
    print("\n=== Cruce de hogares (Sumaria define la población) ===")
    print(f"Hogares en Sumaria:                    {len(ids_sum):,}")
    print(f"Hogares en módulo 100:                 {len(ids_100):,}")
    print(f"  en ambos:                            {len(ids_sum & ids_100):,}")
    print(f"  solo en módulo 100 (saldrán):        {len(ids_100 - ids_sum):,}")
    print(f"  solo en Sumaria (sin módulo 100):    {len(ids_sum - ids_100):,}")
    for nombre in ("m200", "m300"):
        ids = set(tablas[nombre]["id_hogar"])
        print(f"Hogares del {nombre} que no están en Sumaria: {len(ids - ids_sum):,}"
              f" | hogares de Sumaria sin filas en {nombre}: {len(ids_sum - ids):,}")
 
    if "result" in m100.columns:
        fuera = m100[~m100["id_hogar"].isin(ids_sum)]
        print("\nResultado final de la encuesta (RESULT) de los hogares del 100 que NO están en Sumaria:")
        print(fuera["result"].value_counts(dropna=False).to_string())
 
    if "pobreza" in sumaria.columns:
        print("\nCódigos de POBREZA en Sumaria (confirma qué significa cada uno):")
        print(sumaria["pobreza"].value_counts(dropna=False).sort_index().to_string())
 
    print("\n=== Guardando versiones con llaves en data/interim ===")
    for nombre, df in tablas.items():
        print(f"  {guardar(df, nombre)}")
if __name__ == "__main__":
    main()

Leyendo Sumaria-2025.csv ...
  33,702 filas x 162 columnas | hogares distintos: 33,702
  'id_hogar' repetidos: 0  (ok, único)
Leyendo Enaho01-2025-100.csv ...
  44,599 filas x 231 columnas | hogares distintos: 44,599
  'id_hogar' repetidos: 0  (ok, único)
Leyendo Enaho01-2025-200.csv ...
  115,145 filas x 42 columnas | hogares distintos: 33,702
  'id_persona' repetidos: 0  (ok, único)
Leyendo Enaho01A-2025-300.csv ...
  104,446 filas x 343 columnas | hogares distintos: 33,702
  'id_persona' repetidos: 0  (ok, único)

=== Cruce de hogares (Sumaria define la población) ===
Hogares en Sumaria:                    33,702
Hogares en módulo 100:                 44,599
  en ambos:                            33,702
  solo en módulo 100 (saldrán):        10,897
  solo en Sumaria (sin módulo 100):    0
Hogares del m200 que no están en Sumaria: 0 | hogares de Sumaria sin filas en m200: 0
Hogares del m300 que no están en Sumaria: 0 | hogares de Sumaria sin filas en m300: 0

Resultado final de la en

In [43]:
sumaria = pd.read_pickle(INTERIM / "sumaria.pkl")
m100 = pd.read_pickle(INTERIM / "m100.pkl")

print(sumaria["pobreza"].value_counts(dropna=False).sort_index())
print()
fuera = m100[~m100["id_hogar"].isin(sumaria["id_hogar"])]
print(fuera["result"].value_counts(dropna=False))

pobreza
1     1182
2     4922
3    27598
Name: count, dtype: int64

result
7    5976
5    3041
3    1586
4     294
Name: count, dtype: int64


Revisar las salidas
id_hogar es conglome + vivienda + hogar. Identifica un hogar y es la llave que une todas las tablas, varios hogares pueden vivir en una misma vivienda. Para las personas (módulos 200 y 300) existe también id_persona.

In [44]:
df_revisar = pd.read_pickle(INTERIM / "sumaria.pkl")
print("Dimensiones de la tabla:", df_revisar.shape)
display(df_revisar.head())

Dimensiones de la tabla: (33702, 162)


,ano,mes,conglome,vivienda,hogar,ubigeo,dominio,estrato,mieperho,totmieho,...,estrsocial,linpe,linea,pobreza,lineav,pobrezav,factor07,nconglome,sub_conglome,id_hogar
0,2025,01,015009,012,11,010101,4,4,2,2,...,4,"234,982925415039","387,285095214844",3,"659,26025390625",4,"87,2856903076172",07098,00,01500901211
1,2025,01,015009,025,11,010101,4,4,6,6,...,4,"234,982925415039","387,285095214844",3,"696,319396972656",3,"87,2856903076172",07098,00,01500902511
2,2025,01,015009,049,11,010101,4,4,3,3,...,4,"234,982925415039","387,285095214844",3,"721,896911621094",4,"87,2856903076172",07098,00,01500904911
3,2025,01,015009,061,11,010101,4,4,1,1,...,4,"234,982925415039","387,285095214844",3,"705,333862304688",4,"87,2856903076172",07098,00,01500906111
4,2025,01,015009,074,11,010101,4,4,3,3,...,4,"234,982925415039","387,285095214844",3,"654,09912109375",4,"87,2856903076172",07098,00,01500907411
